## TL;DR — Plain English

**What this notebook does:**
Takes you from zero knowledge to a working mental model of AlphaFold 3 in 3 hours.

**By the end, you will understand:**
- What a protein is (amino acid sequence → 3D structure)
- Why predicting the 3D structure from sequence was hard for 60 years
- The complete AF3 architecture: what goes in, what comes out, what each component does
- Why triangle attention (not regular attention) is needed for 3D structure
- How diffusion generates structures from noise
- The three equations that define AF3 mathematically

**You do NOT need to know:** biology, chemistry, structural biology, or advanced math.
You need: Python loops and functions, basic matrix multiply intuition.

**After this notebook:** Go to `07/01_af3_architecture.ipynb` for implementation details.


## Learning Objectives
- [ ] Explain what a protein is and why its 3-D shape matters
- [ ] Identify key structural features: backbone, side-chains, secondary structure
- [ ] Describe what AlphaFold 3 predicts and why it is a breakthrough
- [ ] Map the 20 amino acids to their one-letter codes

## 🧬 Protein Structure & AlphaFold 3 — Concepts for Beginners

| Term | Plain English |
|------|--------------|
| **Protein** | A molecular machine built from a chain of amino acids; it folds into a 3-D shape that determines its function |
| **Residue** | One amino acid unit inside a protein chain (the term used once it is part of the chain) |
| **Fold** | The specific 3-D arrangement a protein adopts; misfolding causes diseases like Alzheimer's |
| **Cα (C-alpha)** | The central carbon atom in every amino acid — connecting Cα atoms gives the protein backbone trace |
| **Backbone** | The repeating N–Cα–C=O chain that forms the protein's structural spine |
| **Domain** | A compact, independently-folding region of a protein (e.g., kinase domain, SH2 domain) |
| **Kinase** | An enzyme that transfers a phosphate group to a substrate; major drug target family |
| **Binding site** | The pocket or surface patch where a protein grabs its partner molecule |
| **Ligand** | A small molecule that binds to a protein (drug, cofactor, metabolite) |
| **Active site** | The specific binding site where a chemical reaction is catalysed |
| **Loop** | A flexible stretch connecting helices or sheets; often forms the binding site |
| **Helix (α-helix)** | A common coiled secondary structure stabilised by backbone hydrogen bonds |

# AlphaFold 3 — Zero to Hero
## Start Here If You Have No Background

This notebook assumes you know **basic Python** but nothing about proteins, biology, or structure prediction.
By the end you will understand the entire AlphaFold 3 problem from the ground up.

**Who this is for:** Anyone starting modules 07 or 10 who finds the other notebooks assume too much.

**Time:** ~3 hours | **No prerequisites beyond Python lists and loops**


---
## Chapter 1: What Is a Protein? (Biology in 10 Minutes)

A protein is a **string of amino acids** that folds into a specific 3D shape.
That shape determines what the protein DOES in your body.

```
Amino acid string: M-V-H-L-T-P-E-E-K-S-A-V-T...
       ↓ (folds spontaneously in milliseconds)
3D structure: [complex tangled shape]
       ↓
Function: carries oxygen (hemoglobin), catalyzes reactions, builds cells
```

**There are only 20 standard amino acids.** Each has a 1-letter code (A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y).


## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- None — this notebook is designed as a fresh start into AlphaFold 3.
- **Optional review:** protein structure basics (PDB, amino acids, secondary structure). *Review: `03_protein_structural_biology/01_protein_structure.ipynb`*

**Quick recap of terms used here:**
- **attention** — mechanism where each element queries all others to decide what to focus on. *Fully covered in `05_deep_learning_finetuning/01_dl_and_finetuning.ipynb`.*

In [ ]:
# The 20 amino acids — learn these, they appear everywhere in AF3 code
amino_acids = {
    'A': ('Alanine',       'nonpolar',  'small',     'CHₓ side chain'),
    'C': ('Cysteine',      'nonpolar',  'small',     'SH — forms disulfide bonds'),
    'D': ('Aspartate',     'charged-',  'small',     'COO⁻ — negative at pH 7'),
    'E': ('Glutamate',     'charged-',  'medium',    'COO⁻ — negative, longer'),
    'F': ('Phenylalanine', 'aromatic',  'large',     'benzyl ring'),
    'G': ('Glycine',       'nonpolar',  'tiny',      'no side chain — max flexibility'),
    'H': ('Histidine',     'aromatic',  'medium',    'pKa≈6, switches charge at pH 6-8'),
    'I': ('Isoleucine',    'nonpolar',  'large',     'branched hydrophobic'),
    'K': ('Lysine',        'charged+',  'long',      'NH₃⁺ — positive'),
    'L': ('Leucine',       'nonpolar',  'large',     'branched hydrophobic'),
    'M': ('Methionine',    'nonpolar',  'medium',    'thioether — start codon AUG'),
    'N': ('Asparagine',    'polar',     'small',     'amide — hydrogen bonds'),
    'P': ('Proline',       'nonpolar',  'rigid',     'ring — breaks helix'),
    'Q': ('Glutamine',     'polar',     'medium',    'amide — hydrogen bonds, longer'),
    'R': ('Arginine',      'charged+',  'long',      'guanidinium — strongest +charge'),
    'S': ('Serine',        'polar',     'small',     'OH — phosphorylation target'),
    'T': ('Threonine',     'polar',     'medium',    'OH — phosphorylation target'),
    'V': ('Valine',        'nonpolar',  'medium',    'branched hydrophobic'),
    'W': ('Tryptophan',    'aromatic',  'largest',   'indole ring — fluorescent'),
    'Y': ('Tyrosine',      'aromatic',  'large',     'phenol — tyrosine kinase target'),
}

# In AF3 code, amino acids are encoded as integers 0-19 + special tokens
AA_TO_INT = {aa: i for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}
INT_TO_AA = {v: k for k, v in AA_TO_INT.items()}

print("The 20 amino acids and their properties:")
print(f"{'AA'} {'Name':<15} {'Class':<12} {'Size':<8} {'Key feature'}")
print("-" * 65)
for aa, (name, cls, size, feat) in amino_acids.items():
    print(f" {aa}  {name:<15} {cls:<12} {size:<8} {feat}")

print(f"
AA_TO_INT encoding used in PyTorch:")
print("  'A' → 0, 'C' → 1, 'D' → 2 ...")
print(f"  Full alphabet: {AA_TO_INT}")
print()

# Key insight: hydrophobic AAs go INSIDE (away from water)
#              charged/polar AAs go OUTSIDE (interact with water)
hydrophobic = [aa for aa, (_, cls, _, _) in amino_acids.items()
               if cls in ('nonpolar', 'aromatic')]
charged     = [aa for aa, (_, cls, _, _) in amino_acids.items()
               if 'charged' in cls or cls == 'polar']
print(f"Hydrophobic (prefer inside): {hydrophobic}")
print(f"Polar/Charged (prefer surface): {charged}")
print()
print("WHY THIS MATTERS FOR AF3:")
print("  Hydrophobic collapse = primary driving force of protein folding")
print("  AF3 learns these preferences from millions of known structures")

> 📋 **Expected output:** A dictionary or table of the 20 amino acids with their one-letter codes, three-letter codes, and properties.
> ✅ You should see 20 entries. If you see an error, check that the cell ran completely.

## 🔧 Common Errors & Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `ModuleNotFoundError: No module named 'Bio'` | Biopython not installed | `!pip install biopython`, restart kernel |
| `KeyError` when accessing amino acid dict | Using 3-letter code instead of 1-letter | Use the 1-letter key (e.g., `'A'` not `'Ala'`) |
| `FileNotFoundError` for PDB file | File path wrong or not downloaded | Check path; use `urllib` to download from RCSB |
| `IndexError` on residue list | Off-by-one: PDB residue numbering starts at 1 | Remember Python lists are 0-indexed |
| `ValueError: could not convert string to float` | Parsing a non-numeric PDB column | Double-check column slicing (PDB is fixed-width) |

## 🌍 Real-World Impact

**AlphaFold has already changed biology:**
- **200M+ structures** predicted and freely available in the AlphaFold Protein Structure Database
- **Drug discovery** — pharmaceutical companies use AF3 predictions to identify drug binding sites and design molecules
- **Neglected diseases** — researchers studying tropical diseases now have structural data that would have taken decades to obtain experimentally
- **Enzyme engineering** — companies design novel enzymes for industrial applications (biofuels, plastics recycling) using AF3-generated structures

**This is why you're learning AF3:** It's not just an academic exercise — it's a tool being used right now to accelerate research and save lives.

## 📚 Further Reading

**If you want to go deeper after this notebook:**
- **AF3 paper:** Abramson et al. (2024) "Accurate structure prediction of biomolecular interactions with AlphaFold 3" — *Nature*. DOI: 10.1038/s41586-024-07487-w
- **AF2 paper (the predecessor):** Jumper et al. (2021) "Highly accurate protein structure prediction with AlphaFold" — *Nature*. DOI: 10.1038/s41586-021-03819-2
- **Protein basics textbook:** Branden & Tooze, "Introduction to Protein Structure" — free chapters at many university libraries
- **Video:** "AlphaFold and the Grand Challenge" — 20-min YouTube overview by the research team
- **Next notebook:** `07_alphafold3_core/01_af3_architecture.ipynb` for the full architecture deep dive

## 🎤 Interview Questions

**Q1 (Easy):** What are the four levels of protein structure (primary through quaternary)?
<details><summary>Answer</summary>
Primary = amino acid sequence. Secondary = local motifs (α-helix, β-sheet). Tertiary = full 3-D fold of one chain. Quaternary = arrangement of multiple chains (subunits) in a complex.
</details>

**Q2 (Easy):** Why does the 3-D structure of a protein matter more than its sequence alone?
<details><summary>Answer</summary>
Function is determined by shape: the geometry of binding sites, catalytic residues, and surface charge. Two proteins with low sequence similarity can share the same fold and function (remote homologues), while a single point mutation can destroy the fold entirely.
</details>

**Q3 (Medium):** What problem does AlphaFold 3 solve, and what makes it harder than typical ML tasks?
<details><summary>Answer</summary>
AF3 predicts the 3-D atomic coordinates of biomolecular complexes (proteins, nucleic acids, ligands) from sequence. The challenge is that the output space is continuous 3-D coordinates of thousands of atoms, the ground truth is sparse (only ~200k experimental structures), and the physics is governed by long-range interactions that simple distance metrics miss.
</details>

**Q4 (Medium):** Explain the role of multiple sequence alignments (MSAs) in structure prediction.
<details><summary>Answer</summary>
MSAs reveal co-evolutionary signal: pairs of residues that mutate together are likely in spatial contact. AF2/AF3 use MSA features to build pair representations that encode these co-evolutionary constraints, which dramatically improve accuracy over single-sequence prediction.
</details>

**Q5 (Hard):** How does AlphaFold 3's diffusion-based approach differ from AlphaFold 2's direct coordinate regression, and what advantage does it provide?
<details><summary>Answer</summary>
AF2 directly regresses atom coordinates through an iterative structure module with IPA. AF3 replaces this with a diffusion model that denoises atom positions from random noise. The advantage is that diffusion naturally produces a distribution of conformations (capturing uncertainty), handles multi-modal outputs (e.g., flexible loops), and scales better to general biomolecular complexes including small molecules and nucleic acids without requiring separate SE(3)-equivariant modules.
</details>

## ✅ Notebook Complete

**You can now:**
- Describe protein structure at all four levels (primary → quaternary)
- Identify the 20 amino acids by one-letter code and chemical property
- Explain what AlphaFold 3 predicts and why structure prediction matters
- Distinguish backbone atoms (N, Cα, C, O) from side-chain atoms

**Next:** → `07_alphafold3_core/01_af3_architecture.ipynb`